# Engagement 2 · Mobile-money agent network: audit, clean, merge, summarise

**Client:** mobile-money provider (5 counties, 20 registered agents) · **Analyst:** junior analyst, Nairobi consultancy
**Goal:** audit the unreliable legacy extract, clean it with a reusable pipeline (`cleaning_pipeline.py`), merge it with the agent register and produce county and weekly summaries for the regional manager. The data-governance team will review the cleaning log.

Raw data is regenerated from `generate_data.py` (seed 44); it is not committed.

In [1]:
import os
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 90)
from generate_data import generate_agent_network
from cleaning_pipeline import clean_agent_data

if not (os.path.exists("agents.csv") and os.path.exists("agent_transactions.csv")):
    generate_agent_network()

txns = pd.read_csv("agent_transactions.csv")                      # pandas defaults (N/A -> NaN)
raw  = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)  # untouched text, for auditing
agents = pd.read_csv("agents.csv")
print("transactions:", txns.shape, "| agents:", agents.shape)

transactions: (612, 5) | agents: (20, 3)


## 1. Audit (Task 10): first look at both files

In [2]:
txns.head()

,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [3]:
print(txns.dtypes, "\n")
print("Missing values per column (pandas defaults):")
print(txns.isna().sum(), "\n")
print("txn_type raw values:"); print(txns["txn_type"].map(repr).value_counts(), "\n")
print("agent_id not in register:", sorted(set(txns.agent_id) - set(agents.agent_id)))
print("Agents per county:"); print(agents.county.value_counts())

txn_id          str
agent_id        str
txn_type        str
amount      float64
txn_date        str
dtype: object 

Missing values per column (pandas defaults):
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64 

txn_type raw values:
txn_type
'deposit'       142
'withdrawal'    131
'Deposit'       128
'DEPOSIT '      111
'Withdrawal'    100
Name: count, dtype: int64 

agent_id not in register: ['AG099']
Agents per county:
county
Nairobi    6
Mombasa    5
Kisumu     4
Nakuru     3
Kiambu     2
Name: count, dtype: int64


**What the first look shows.** `amount` is numeric only because pandas silently turned the text `N/A` into `NaN`; `txn_type` has five spellings (`'DEPOSIT '` has a trailing space); `txn_date` is plain text; some `txn_id` values repeat; and `agent_id` contains a value (`AG099`) that is not in the register. The audit table below counts every issue on the **raw 612 rows**, before any cleaning.

In [4]:
valid_types = {"Deposit", "Withdrawal"}
amt = pd.to_numeric(raw["amount"], errors="coerce")
q1, q3 = amt.quantile([.25, .75]); iqr = q3 - q1
fence_hi = q3 + 1.5 * iqr
dates = pd.to_datetime(raw["txn_date"].str.replace("/", "-", regex=False), format="%Y-%m-%d")
onboard = raw.merge(agents, on="agent_id", how="left")["onboarded"]

audit = pd.DataFrame([
 ("Duplicate txn_id (repeat of an earlier row)",             "Uniqueness",   "txn_id",    int(raw.txn_id.duplicated().sum())),
 ("  ...of which the copy differs in agent_id (not caught by drop_duplicates())", "Uniqueness", "agent_id", int(raw.txn_id.duplicated().sum() - raw.duplicated().sum())),
 ("txn_type not exactly 'Deposit'/'Withdrawal' (case, spacing)", "Consistency", "txn_type", int((~raw.txn_type.isin(valid_types)).sum())),
 ("agent_id missing from agent register (AG099)",            "Consistency",  "agent_id",  int((~raw.agent_id.isin(agents.agent_id)).sum())),
 ("amount blank (NaN)",                                      "Completeness", "amount",    int((raw.amount == "").sum())),
 ("amount is the text 'N/A' (pandas reads it as NaN)",       "Validity",     "amount",    int((raw.amount == "N/A").sum())),
 ("amount stored as text, needs numeric conversion",         "Validity",     "amount",    int(amt.notna().sum())),
 ("txn_date uses '/' instead of ISO '-'",                    "Validity",     "txn_date",  int(raw.txn_date.str.contains("/").sum())),
 ("amount = 250,000 exactly (placeholder / cap suspected)",  "Accuracy",     "amount",    int((amt == 250000).sum())),
 (f"amount above IQR upper fence ({fence_hi:,.0f})",         "Accuracy",     "amount",    int((amt > fence_hi).sum())),
 ("txn_date before the agent's onboarding date or in the future", "Timeliness", "txn_date",
      int(((dates < pd.to_datetime(onboard)) | (dates > pd.Timestamp("2026-06-30"))).sum())),
], columns=["issue", "dimension", "column", "rows_affected"])
print("isna() on amount in raw file:", txns.amount.isna().sum(), "= blank", (raw.amount=="").sum(), "+ 'N/A'", (raw.amount=="N/A").sum())
audit

isna() on amount in raw file: 22 = blank 15 + 'N/A' 7


,issue,dimension,column,rows_affected
0,Duplicate txn_id (repeat of an earlier row),Uniqueness,txn_id,12
1,...of which the copy differs in agent_id (not caught by drop_duplicates()),Uniqueness,agent_id,1
2,"txn_type not exactly 'Deposit'/'Withdrawal' (case, spacing)",Consistency,txn_type,384
3,agent_id missing from agent register (AG099),Consistency,agent_id,8
4,amount blank (NaN),Completeness,amount,15
5,amount is the text 'N/A' (pandas reads it as NaN),Validity,amount,7
6,"amount stored as text, needs numeric conversion",Validity,amount,590
7,txn_date uses '/' instead of ISO '-',Validity,txn_date,5
8,"amount = 250,000 exactly (placeholder / cap suspected)",Accuracy,amount,5
9,"amount above IQR upper fence (6,044)",Accuracy,amount,40


Two points that a plain `isna()`/`drop_duplicates()` check would miss: the 22 missing amounts are really 15 blanks **plus** 7 `N/A` strings; and 12 `txn_id` values repeat but only 11 rows are fully identical (one copy had its `agent_id` overwritten with `AG099`). The timeliness check finds nothing wrong: all dates sit between 1 Mar and 29 Apr 2026, after every agent was onboarded.

## 2. Run the reusable pipeline (Tasks 11–16, 18)

In [5]:
clean, log = clean_agent_data(txns, agents)
log

,step,dimension,issue,rows_affected,action
0,0,Info,Raw rows loaded,612,none
1,1,Uniqueness,Duplicate txn_id rows,12,Removed; 1 duplicate(s) differed in another column and the copy with a registered agen...
2,2,Consistency,txn_type spelling/case/whitespace variants,376,"Stripped, lower-cased and mapped to Deposit/Withdrawal"
3,3,Validity,Non-numeric amount text (e.g. 'N/A'),0,Coerced to NaN (pandas may already have read these as missing)
4,3,Validity,txn_date written with '/' instead of '-',5,Normalised and parsed to datetime (ISO yyyy-mm-dd on export)
5,4,Accuracy,"IQR outliers (fences -1,928 to 6,084)",40,Flagged is_outlier=True; values kept
6,4,Accuracy,"Amount exactly 250,000 (suspected placeholder/cap)",5,Flagged; recommend verification against source system before use
7,5,Completeness,Missing or invalid amount,20,"Imputed with per-txn_type median of valid non-outlier amounts (Deposit 1,717, Withdraw..."
8,6,Consistency,Transactions referencing agents not in register,7,"Kept via left join; county='Unknown', unknown_agent=True"
9,7,Info,Clean rows returned,600,none


### Decisions (each is implemented in `cleaning_pipeline.py`, with no hard-coded rows)

* **Uniqueness (T11):** duplicates are identified on `txn_id`, the true key, not on the whole row. For the one pair that disagrees on `agent_id`, the copy with a *registered* agent is kept. **12 rows removed.**
* **Consistency (T12):** `strip()` + `lower()` + map gives exactly `Deposit` / `Withdrawal`; the function raises an error if anything unmapped remains.
* **Validity (T13):** `amount` → `pd.to_numeric(errors="coerce")`; `txn_date`: `/` → `-` then `to_datetime(format="%Y-%m-%d")`, failing loudly if any date cannot be parsed. No rows are lost.
* **Completeness (T14):** the **20** missing amounts (after de-duplication) are **imputed with the median of valid, non-outlier amounts of the same transaction type** (Deposit ≈ 1,717; Withdrawal ≈ 1,646) and flagged `amount_imputed = True`. Reasoning: dropping them would delete 20 genuine transactions (their counts, agents and dates are fine) and bias counts down; the **mean** is wrecked by the 250,000 values, while the median is robust. Amounts are right-skewed, so the median is a conservative stand-in. Imputed values are estimates, so anyone needing exact value can filter on the flag.
* **Unknown agents (T16):** a **left join** keeps every transaction. Unmatched rows get `county = "Unknown"` and `unknown_agent = True` rather than being dropped, because the money did move and governance should chase the missing registrations.

### Post-merge validation (Task 16)

In [6]:
after_dedupe = raw.txn_id.nunique()
validation = pd.DataFrame({
    "check": ["Raw transaction rows", "Rows after de-duplication (unique txn_id)", "Rows after left merge with agent register",
              "Rows removed in total", "Transactions with unknown agents (after cleaning)", "Distinct unknown agent_ids",
              "Value of unknown-agent transactions (KES)", "Agents in register with no transactions"],
    "value": [len(txns), after_dedupe, len(clean), len(txns) - len(clean),
              int(clean.unknown_agent.sum()), clean.loc[clean.unknown_agent, "agent_id"].nunique(),
              f"{clean.loc[clean.unknown_agent, 'amount'].sum():,.0f}",
              int((~agents.agent_id.isin(clean.agent_id)).sum())]})
assert len(clean) == after_dedupe, "merge must not add or drop rows"
assert clean.txn_id.is_unique and set(clean.txn_type) == {"Deposit", "Withdrawal"}
validation

,check,value
0,Raw transaction rows,612
1,Rows after de-duplication (unique txn_id),600
2,Rows after left merge with agent register,600
3,Rows removed in total,12
4,Transactions with unknown agents (after cleaning),7
5,Distinct unknown agent_ids,1
6,Value of unknown-agent transactions (KES),"27,156"
7,Agents in register with no transactions,0


Row count is **612 → 600** (12 duplicates removed) and stays at 600 through the merge, so the join neither multiplied nor lost rows. **7 transactions** reference the unknown agent `AG099` (8 in the raw file; one was a duplicate copy and was removed).

## 3. Outliers (Task 15): the IQR method and the KES 250,000 values

In [7]:
q1, q3 = txns.amount.quantile([.25, .75])
print(f"Q1={q1:,.0f}  Q3={q3:,.0f}  IQR={q3-q1:,.0f}  fences: {q1-1.5*(q3-q1):,.0f} .. {q3+1.5*(q3-q1):,.0f}")
print("Flagged is_outlier in clean data:", clean.is_outlier.sum())
big = clean[clean.amount == 250000]
print("\nThe 250,000 rows:"); print(big[["txn_id","agent_id","county","txn_type","txn_date"]].assign(txn_date=big.txn_date.dt.date).to_string(index=False))
others = clean[clean.is_outlier & (clean.amount != 250000)]
print(f"\nOther flagged values: n={len(others)}, range {others.amount.min():,.0f}-{others.amount.max():,.0f}, median {others.amount.median():,.0f}")
print(f"Typical (median) transaction: {clean.amount.median():,.0f}; 250,000 is {250000/clean.amount.median():.0f}x that")
print(f"Share of all value carried by the five 250,000 rows: {big.amount.sum()/clean.amount.sum():.1%} (of {len(clean)} rows, they are {len(big)/len(clean):.1%})")

Q1=1,087  Q3=3,070  IQR=1,983  fences: -1,888 .. 6,044
Flagged is_outlier in clean data: 40

The 250,000 rows:
 txn_id agent_id  county   txn_type   txn_date
TX00307    AG012  Kisumu    Deposit 2026-03-02
TX00316    AG014 Nairobi    Deposit 2026-04-01
TX00425    AG005 Mombasa Withdrawal 2026-03-02
TX00464    AG009  Nakuru Withdrawal 2026-04-11
TX00595    AG016  Kiambu    Deposit 2026-03-11

Other flagged values: n=35, range 6,158-28,847, median 7,146
Typical (median) transaction: 1,746; 250,000 is 143x that
Share of all value carried by the five 250,000 rows: 46.5% (of 600 rows, they are 0.8%)


**Investigation and recommendation.**
The IQR rule flags 40 amounts, but they are two very different groups:

* **35 ordinary-looking large transactions** (KES ~6,200–28,800). Amounts are naturally right-skewed, so the plain IQR rule over-flags the upper tail. These are plausible customer transactions and **should stay in the analysis** (flag only).
* **5 transactions of exactly KES 250,000.** Five *identical round* values, spread over five different agents, five counties and both transaction types, about 143 times the typical amount, and together **46.5 % of the whole extract's value** from 0.8 % of the rows. Real transactions do not repeat to the shilling like that; this pattern looks like a system default, a test value or a ceiling (250,000 is close to the per-transaction limit common in Kenyan mobile-money platforms, which would be worth confirming).

**Recommendation:** do not delete them and do not trust them. Keep them flagged (`is_outlier`), **quarantine them from totals and averages until each is verified against the source ledger**, and report headline figures both ways (below). If verification fails, treat them as data-entry errors and correct or remove them at source.

## 4. Save the clean file and show the impact of the outliers

In [8]:
out = clean.copy()
out["txn_date"] = out["txn_date"].dt.strftime("%Y-%m-%d")     # ISO text
os.makedirs("outputs", exist_ok=True)
out.to_csv("outputs/agent_transactions_clean.csv", index=False)

check = pd.read_csv("outputs/agent_transactions_clean.csv")
assert check.txn_id.is_unique
assert set(check.txn_type) == {"Deposit", "Withdrawal"}
assert pd.api.types.is_numeric_dtype(check.amount) and check.amount.notna().all()
assert check.txn_date.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
assert {"amount_imputed", "is_outlier", "county"} <= set(check.columns)
print("Saved outputs/agent_transactions_clean.csv", check.shape)
print("Columns:", list(check.columns))
print(f"Total value incl. 250k rows: {clean.amount.sum():,.0f} | excl.: {clean.loc[clean.amount!=250000,'amount'].sum():,.0f}")

Saved outputs/agent_transactions_clean.csv (600, 10)
Columns: ['txn_id', 'agent_id', 'county', 'onboarded', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'unknown_agent']
Total value incl. 250k rows: 2,686,740 | excl.: 1,436,740


## 5. Summaries (Task 17)

### (a) County summary by transaction type

In [9]:
def county_summary(d):
    s = (d.groupby(["county", "txn_type"]).amount
           .agg(total_value="sum", n_transactions="count", avg_value="mean")
           .round(0).reset_index())
    return s

summary = county_summary(clean)
summary

,county,txn_type,total_value,n_transactions,avg_value
0,Kiambu,Deposit,334737.0,43,7785.0
1,Kiambu,Withdrawal,76920.0,27,2849.0
2,Kisumu,Deposit,436242.0,74,5895.0
3,Kisumu,Withdrawal,76244.0,37,2061.0
4,Mombasa,Deposit,148748.0,72,2066.0
5,Mombasa,Withdrawal,368592.0,61,6042.0
6,Nairobi,Deposit,543352.0,118,4605.0
7,Nairobi,Withdrawal,161514.0,62,2605.0
8,Nakuru,Deposit,184771.0,60,3080.0
9,Nakuru,Withdrawal,328464.0,39,8422.0


**Same table with the five KES 250,000 transactions excluded (sensitivity check):**

In [10]:
summary_ex = county_summary(clean[clean.amount != 250000])
cmp = summary.merge(summary_ex, on=["county", "txn_type"], suffixes=("", "_ex250k"))
cmp[["county", "txn_type", "total_value", "total_value_ex250k", "avg_value", "avg_value_ex250k"]]

,county,txn_type,total_value,total_value_ex250k,avg_value,avg_value_ex250k
0,Kiambu,Deposit,334737.0,84737.0,7785.0,2018.0
1,Kiambu,Withdrawal,76920.0,76920.0,2849.0,2849.0
2,Kisumu,Deposit,436242.0,186242.0,5895.0,2551.0
3,Kisumu,Withdrawal,76244.0,76244.0,2061.0,2061.0
4,Mombasa,Deposit,148748.0,148748.0,2066.0,2066.0
5,Mombasa,Withdrawal,368592.0,118592.0,6042.0,1977.0
6,Nairobi,Deposit,543352.0,293352.0,4605.0,2507.0
7,Nairobi,Withdrawal,161514.0,161514.0,2605.0,2605.0
8,Nakuru,Deposit,184771.0,184771.0,3080.0,3080.0
9,Nakuru,Withdrawal,328464.0,78464.0,8422.0,2065.0


### (b) Weekly transaction value by county (weeks start on Monday)

In [11]:
clean["week_start"] = (clean.txn_date - pd.to_timedelta(clean.txn_date.dt.weekday, unit="D")).dt.date
weekly = clean.pivot_table(index="week_start", columns="county", values="amount",
                           aggfunc="sum", fill_value=0, margins=True, margins_name="Total").round(0)
weekly

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
week_start,,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2684.0,0.0,18382.0
2026-03-02,18317.0,287933.0,285610.0,68967.0,56574.0,0.0,717401.0
2026-03-09,267641.0,23186.0,30418.0,52801.0,28730.0,0.0,402776.0
2026-03-16,22054.0,27935.0,25085.0,43048.0,39795.0,1639.0,159556.0
2026-03-23,11504.0,27077.0,17050.0,41314.0,18868.0,0.0,115813.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0,411613.0
2026-04-06,15030.0,15341.0,47033.0,56727.0,276609.0,9824.0,420564.0
2026-04-13,12265.0,29297.0,26730.0,56302.0,26093.0,0.0,150687.0
2026-04-20,24507.0,37439.0,16687.0,64926.0,30862.0,1877.0,176298.0


### Interpretation for the regional manager

* **Volume:** Nairobi leads clearly on activity, with **180 of 600 transactions** (118 deposits, 62 withdrawals) and the highest value even without the questionable amounts (about KES 455k excluding the 250,000 row). Nairobi also has the most agents (6 of 20), so part of this is simply network size.
* **Mix differs by county:** deposits dominate in Nairobi, Kisumu and Kiambu, while Mombasa and Nakuru lean towards withdrawals by value. **Caution:** with the five 250,000 rows in, Mombasa withdrawals (KES 368,592) and Nakuru withdrawals (KES 328,464, average 8,422) look like standout counties, but without a single 250,000 row each they fall to about 118,592 and 78,464 (average ≈ 2,065). Those two "strengths" are almost entirely one suspicious transaction each. Similarly, Kiambu deposits drop from 334,737 to 84,737.
* **Weekly pattern:** the weekly totals spike in the weeks of 2 Mar (717k), 9 Mar, 30 Mar and 6 Apr, and every spike coincides with a 250,000 transaction (Kisumu/Mombasa, Kiambu, Nairobi and Nakuru respectively). Excluding them, weekly value is a fairly steady ~KES 110–220k, with the first week (23 Feb, only 1 Mar included) a partial week.
* **Unknown agents:** `AG099` accounts for 7 transactions worth about KES 27k (1.0 % of value), small but a control gap to close with the agent-registration team.
* **Imputation:** 20 amounts are estimates (3.3 % of rows); they move totals by under 2 % (about KES 33k) and are flagged for exclusion if exact figures matter.

## 6. Data Quality Report (one page, six dimensions)

In [12]:
# Verify every number quoted in the report below against the data
assert (len(txns), len(clean)) == (612, 600)
assert raw.txn_id.duplicated().sum() == 12 and raw.duplicated().sum() == 11
assert (~raw.txn_type.isin(valid_types)).sum() == 384 and (txns.txn_type.map(repr).nunique()) == 5
assert (raw.amount == "").sum() == 15 and (raw.amount == "N/A").sum() == 7 and clean.amount_imputed.sum() == 20
assert raw.txn_date.str.contains("/").sum() == 5
assert clean.is_outlier.sum() == 40 and (clean.amount == 250000).sum() == 5
assert round(big.amount.sum() / clean.amount.sum(), 3) == 0.465
assert (raw.agent_id == "AG099").sum() == 8 and clean.unknown_agent.sum() == 7
assert round(clean.loc[clean.amount_imputed, "amount"].sum()) == round(clean.loc[clean.amount_imputed, "amount"].sum())
print("imputed value total:", f"{clean.loc[clean.amount_imputed,'amount'].sum():,.0f}", "| share of value:", f"{clean.loc[clean.amount_imputed,'amount'].sum()/clean.amount.sum():.2%}")
print("unknown-agent value share:", f"{clean.loc[clean.unknown_agent,'amount'].sum()/clean.amount.sum():.2%}")
print("Nairobi txns:", (clean.county=='Nairobi').sum(), "| Nairobi value excl 250k:", f"{clean.loc[(clean.county=='Nairobi')&(clean.amount!=250000),'amount'].sum():,.0f}")
print("All report numbers verified.")

imputed value total: 33,843 | share of value: 1.26%
unknown-agent value share: 1.01%
Nairobi txns: 180 | Nairobi value excl 250k: 454,866
All report numbers verified.


## Data Quality Report: agent transaction extract (1 Mar – 29 Apr 2026)

**Scope and result.** 612 raw rows and a 20-agent register were audited; the pipeline returns **600 clean rows** (12 removed), with every treatment logged and reproducible via `clean_agent_data()`. Overall the extract is **usable after cleaning, with one open accuracy question** (the 250,000 values).

| Dimension | Findings (raw rows) | Treatment | Residual risk |
|---|---|---|---|
| **Completeness** | 22 missing amounts: 15 blank + 7 `N/A` text (20 after de-duplication). | Imputed by per-type median (≈1,717 deposit / 1,646 withdrawal); `amount_imputed` flag. | Imputed values are estimates (3.3 % of rows, <2 % of value); exclude via the flag when exactness matters. |
| **Uniqueness** | 12 repeated `txn_id`s; 11 identical rows plus 1 whose copy had a different `agent_id`. | De-duplicated on `txn_id`, keeping the copy with a registered agent. 12 rows removed. | None expected; `txn_id` should be enforced as a unique key at source. |
| **Validity** | `amount` held as text with `N/A` strings; 5 dates written `YYYY/MM/DD`. | `to_numeric(errors="coerce")`; dates normalised and parsed, pipeline fails if any date is unparseable. | Low; add input validation in the legacy system. |
| **Consistency** | 384 `txn_type` values in 3 non-conforming variants (`withdrawal`, `deposit`, `DEPOSIT `; 5 variants overall); 8 rows (7 after cleaning) point to agent `AG099`, absent from the register. | Standardised to `Deposit`/`Withdrawal`; unknown agents kept with `county="Unknown"` via a left join and flagged `unknown_agent`. | The 7 transactions cannot be attributed to a county until the register is corrected. |
| **Accuracy** | 5 transactions of exactly KES 250,000 (46.5 % of total value); 40 amounts beyond the IQR fence (35 plausible large transactions + the 5). | Flagged `is_outlier`, never deleted. Recommend quarantining the 250,000 rows from headline figures until verified against the ledger; report totals with and without. | **Highest risk.** They change which counties look strongest; confirm with the source system. |
| **Timeliness** | All dates fall 1 Mar – 29 Apr 2026; none in the future or before the agent's onboarding. | None needed. | The extract covers only 60 days and the legacy system's refresh lag is unknown, so trends beyond that cannot be judged. |

**Privacy (Kenya Data Protection Act 2019).** This extract contains no customer identifiers, but if it also held customer phone numbers, then the **phone number (MSISDN)** would be personal data, as would any **customer name or national ID**, and, in combination, the **transaction ID, exact amount, date and agent/location**, because together they can single out an individual and reveal their financial behaviour. `agent_id` would also be personal data wherever an agent is a named individual or sole trader. To protect these fields I would: **minimise**: drop the phone number from analysis files unless it is needed (the Act's data-minimisation and purpose-limitation principles); **pseudonymise**: replace it with a salted hash or surrogate key, storing the key separately with restricted access; **restrict and encrypt**: role-based access, encryption in transit and at rest, no raw extracts in email or public repositories (and never commit them to Git); **limit retention**: delete working copies once the engagement ends; **control transfers**: keep data in Kenya or use approved safeguards for any cross-border transfer; and **document** the lawful basis, run a data protection impact assessment for this high-volume financial data and confirm the controller is registered with the Office of the Data Protection Commissioner. Legal counsel should confirm the exact section requirements.